In [26]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


### 1. Chargement des données et suppression des doublons

**Objectif :** Charger le jeu de données brut en gérant l'ensemble des représentations de données manquantes textuelles, puis nettoyer les doublons stricts.

#### Opérations réalisées :
1. **Normalisation des valeurs nulles (`na_values`) :** 
   - Définition d'une liste exhaustive `na_list` regroupant toutes les variations de chaînes textuelles représentant des données manquantes (`'NaN'`, `'NULL'`, `'None'`, espaces vides `''`, etc.).
   - Chargement du fichier CSV via `pd.read_csv()` avec l'option `skipinitialspace=True` pour éliminer automatiquement les espaces parasites au début des champs texte.
2. **Déduplication stricte (`drop_duplicates`) :**
   - Identification et comptage des lignes exactement identiques (`df.duplicated().sum()`).
   - Suppression des doublons stricts pour garantir l'unicité des enregistrements et éviter d'impacter les statistiques descriptives ultérieures.

---

In [27]:

na_list = ['NaN', 'NaN ', 'null', 'NULL', 'None', ' ', 'N/A', '']
file_path = "../data/dataset.csv"
df = pd.read_csv(file_path, na_values=na_list, skipinitialspace=True)
print("=== APERÇU INITIAL ===")
print(f"Dimensions initiales : {df.shape}")
nb_doublons = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"Doublons stricts supprimés : {nb_doublons}")

=== APERÇU INITIAL ===
Dimensions initiales : (45593, 20)
Doublons stricts supprimés : 0


### 2. Nettoyage du texte parasite et conversion des types de données

**Objectif :** Standardiser les variables textuelles, extraire les valeurs numériques polluées par des suffixes et cast les colonnes vers leurs types de données appropriés (`float`, `datetime`, `date`).

#### Opérations réalisées :
1. **Normalisation du texte (`str.strip()`) :**
   - Suppression des espaces invisibles en début et fin de chaîne sur toutes les colonnes de type `object`.
   - Conversion explicite des chaînes résiduelles `'nan'` et `'None'` générées par le transtypage en véritables valeurs nulles Pandas (`np.nan`).
2. **Nettoyage des suffixes parasites :**
   - **`Time_taken(min)` :** Retrait du suffixe `"(min)"` pour ne conserver que la valeur numérique brute du temps de livraison.
   - **`Weatherconditions` :** Retrait du terme `"conditions"` (ex: `"Sunnyconditions"` $\rightarrow$ `"Sunny"`).
3. **Conversion numérique (`pd.to_numeric`) :**
   - Transtypage des colonnes quantitatives (coordonnées GPS, âge, notes, livraisons multiples, durée de livraison) en type numérique.
   - Gestion des échecs de conversion via `errors="coerce"` pour transformer toute valeur corrompue en `NaN`.
4. **Parsing et formatage temporel :**
   - **`Order_Date` :** Conversion au format Date via `pd.to_datetime()` avec détection prioritaire du jour (`dayfirst=True`), suivie d'une extraction de la date seule (`.dt.date`) pour éliminer l'horodatage indésirable `00:00:00`.
   - **`Time_Orderd` & `Time_Order_picked` :** Parsing des heures d'enregistrement et de prise en charge au format strict `HH:MM:SS`.

---

In [28]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': np.nan, 'None': np.nan})

if 'Time_taken(min)' in df.columns:
    df["Time_taken(min)"] = df['Time_taken(min)'].astype(str).str.replace("(min)", "", regex=False).str.strip()

if 'Weatherconditions' in df.columns:
    df["Weatherconditions"] = df['Weatherconditions'].astype(str).str.replace("conditions", "", regex=False).str.strip()
val_num = ["Time_taken(min)", "Restaurant_latitude", "Restaurant_longitude", 
           "Delivery_location_latitude", "Delivery_location_longitude", 
           "multiple_deliveries", "Delivery_person_Age", "Delivery_person_Ratings"]

for col in val_num:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
if 'Order_Date' in df.columns:
    df["Order_Date"] = pd.to_datetime(df["Order_Date"], dayfirst=True, errors="coerce").dt.date
if 'Time_Orderd' in df.columns:
    df["Time_Orderd"] = pd.to_datetime(df["Time_Orderd"], format="%H:%M:%S", errors="coerce")
if 'Time_Order_picked' in df.columns:
    df["Time_Order_picked"] = pd.to_datetime(df["Time_Order_picked"], format="%H:%M:%S", errors="coerce")

print("Conversion des types terminée.")

Conversion des types terminée.


/tmp/ipykernel_1315096/1208969209.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


### 3. Traitement des valeurs aberrantes métiers (Domain Knowledge Rules)

**Objectif :** Identifier et invalider les erreurs de saisie ou bugs système sur les variables géographiques, démographiques et d'évaluation en les basculant en `NaN` avant la phase d'imputation.

#### Opérations réalisées :
1. **Validation et filtrage des coordonnées GPS :**
   - **Correction des signes :** Passage de l'ensemble des coordonnées GPS en valeur absolue (`.abs()`) pour corriger les inversions de signe fortuites.
   - **Détection des anomalies physiques :** Neutralisation des points au centre nul $(0,0)$ ainsi que des latitudes hors de la plage validée $[0, 90]^\circ$ et longitudes hors $[0, 180]^\circ$.
   - **Action :** Les coordonnées détectées comme invalides sont remplacées par `np.nan` afin d'être recalculées ultérieurement par la médiane géographique de leur ville respective.
2. **Filtrage des bornes d'âge (`Delivery_person_Age`) :**
   - Application de la règle métier interdisant un âge de livreur hors de la tranche légale/opérationnelle **[18, 65] ans**.
   - Conversion des valeurs aberrantes en `np.nan`.
3. **Filtrage des bornes d'évaluation (`Delivery_person_Ratings`) :**
   - Conservation stricte de l'intervalle de notation de la plateforme **[1.0, 5.0]**.
   - Basculement de toute note inférieure à $1.0$ ou supérieure à $5.0$ en `np.nan`.

---

In [29]:

gps_cols = ["Restaurant_latitude", "Restaurant_longitude", "Delivery_location_latitude", "Delivery_location_longitude"]
df[gps_cols] = df[gps_cols].abs()
invalid_gps = (
    (df["Restaurant_latitude"] == 0) | (df["Restaurant_latitude"] > 90) |
    (df["Restaurant_longitude"] == 0) | (df["Restaurant_longitude"] > 180) |
    (df["Delivery_location_latitude"] == 0) | (df["Delivery_location_latitude"] > 90) |
    (df["Delivery_location_longitude"] == 0) | (df["Delivery_location_longitude"] > 180)
)
df.loc[invalid_gps, gps_cols] = np.nan
print(f"Lignes avec coordonnées GPS aberrantes/invalides remises à NaN : {invalid_gps.sum()}")
if 'Delivery_person_Age' in df.columns:
    invalid_age = (df['Delivery_person_Age'] < 18) | (df['Delivery_person_Age'] > 65)
    df.loc[invalid_age, 'Delivery_person_Age'] = np.nan
    print(f"ages hors bornes (18-65 ans) remis à NaN : {invalid_age.sum()}")
if 'Delivery_person_Ratings' in df.columns:
    invalid_ratings = (df['Delivery_person_Ratings'] < 1.0) | (df['Delivery_person_Ratings'] > 5.0)
    df.loc[invalid_ratings, 'Delivery_person_Ratings'] = np.nan
    print(f"Notes hors bornes (1-5) remises à NaN : {invalid_ratings.sum()}")
print(df.isnull().sum())

Lignes avec coordonnées GPS aberrantes/invalides remises à NaN : 3640
ages hors bornes (18-65 ans) remis à NaN : 38
Notes hors bornes (1-5) remises à NaN : 53
ID                                0
Delivery_person_ID                0
Delivery_person_Age            1892
Delivery_person_Ratings        1961
Restaurant_latitude            3640
Restaurant_longitude           3640
Delivery_location_latitude     3640
Delivery_location_longitude    3640
Order_Date                        0
Time_Orderd                    1731
Time_Order_picked                 0
Weatherconditions                 0
Road_traffic_density            601
Vehicle_condition                 0
Type_of_order                     0
Type_of_vehicle                   0
multiple_deliveries             993
Festival                        228
City                           1200
Time_taken(min)                   0
dtype: int64


### 4. Imputation contextuelle des coordonnées GPS (Stratégie en cascade)

**Objectif :** Reconstruire les coordonnées géographiques manquantes ou préalablement invalidées en exploitant la cohérence spatiale locale par ville et contexte météo, tout en garantissant l'absence totale de valeurs nulles finales.

#### Opérations réalisées :
1. **Implémentation d'un système de secours (*Fallback*) à 3 niveaux :**
   - **Niveau 1 (Contexte fin - `City` + `Weatherconditions`) :** Remplissage prioritaire des coordonnées GPS manquantes par la **médiane** des livraisons associées à la même ville et aux mêmes conditions météorologiques.
   - **Niveau 2 (Contexte régional - `City`) :** Si un groupe météo local ne contient aucune référence valide, la **médiane** calculée sur l'ensemble de la ville prend le relais pour positionner le point au centre de gravité urbain.
   - **Niveau 3 (Secours global - `mean()`) :** En dernier recours, imputation par la **moyenne globale** du dataset pour garantir qu'aucun `NaN` ne subsiste sur la géolocalisation.
2. **Utilisation de la Médiane vs Mode/Moyenne :**
   - Emploi privilégié de la **médiane** pour les niveaux 1 et 2 afin de neutraliser l'impact des valeurs extrêmes sur des variables continues (latitudes/longitudes).
3. **Compatibilité Pandas 2.0+ :**
   - Réaffectation explicite via `df[col] = df[col].fillna(...)` sans usage d'arguments obsolètes (`inplace=True`), éliminant tout risque d'avertissement `ChainedAssignmentError`.

---

In [ ]:

gps_cols = ["Restaurant_latitude", "Restaurant_longitude", "Delivery_location_latitude", "Delivery_location_longitude"]
for col in gps_cols:
    if col in df.columns: 
        med_city_weather = df.groupby(['City', 'Weatherconditions'])[col].transform('median')
        df[col] = df[col].fillna(med_city_weather)
        med_city = df.groupby('City')[col].transform('median')
        df[col] = df[col].fillna(med_city)
        df[col] = df[col].fillna(df[col].median())

print("Coordonnées GPS imputées par (City + Weatherconditions).")




Coordonnées GPS imputées par (City + Weatherconditions).


### 5. Imputation contextuelle des conditions météorologiques (`Weatherconditions`)

**Objectif :** Imputer les valeurs manquantes de la variable catégorielle `Weatherconditions` en s'appuyant sur la cohérence spatio-temporelle de la météo (une ville partage généralement des conditions identiques sur une même journée), à l'aide d'une stratégie en cascade à 3 niveaux.

#### Opérations réalisées :
1. **Implémentation d'une cascade d'imputation par le Mode (*Fallback*) :**
   - **Niveau 1 (Contexte spatio-temporel - `City` + `Order_Date`) :** Remplissage prioritaire par le **mode** (la météo la plus fréquente) observée dans la même ville le même jour.
   - **Niveau 2 (Contexte urbain - `City`) :** Si une journée entière ne contient aucune donnée météo pour une ville donnée, le **mode** global de la ville (toutes dates confondues) est appliqué en second recours.
   - **Niveau 3 (Secours global) :** Remplissage des éventuelles valeurs manquantes restantes par le **mode global** de l'ensemble du dataset (la météo la plus représentative de la base).
2. **Utilisation du Mode (`mode()[0]`) :**
   - Emploi du mode statistique, adapté aux variables catégorielles/textuelles, avec extraction sécurisée de la première valeur (`[0]`) après vérification de l'existence d'un mode (`if not x.mode().empty`).
3. **Alignement et assignation explicite :**
   - Utilisation de `transform()` pour conserver l'indexation d'origine du DataFrame et réaffectation directe `df['Weatherconditions'] = ...` pour respecter les normes de Pandas 2.0+ (*Copy-on-Write*).

---

In [ ]:
if 'Weatherconditions' in df.columns:
    mode_city_date = df.groupby(['City', 'Order_Date'])['Weatherconditions'].transform(
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
    df['Weatherconditions'] = df['Weatherconditions'].fillna(mode_city_date)
    

    mode_city = df.groupby('City')['Weatherconditions'].transform(
        lambda x: x.mode()[0] if not x.mode().empty else np.nan
    )
    df['Weatherconditions'] = df['Weatherconditions'].fillna(mode_city)

    df['Weatherconditions'] = df['Weatherconditions'].fillna(df['Weatherconditions'].mode()[0])
    
    print("'Weatherconditions' imputée par (City + Order_Date).")

'Weatherconditions' imputée par (City + Order_Date).


### Traitement et imputation séquentielle de `Order_Date`

**Objectif :** Reconstituer les dates de commande manquantes en exploitant la séquence temporelle des livraisons par livreur, précédée d'un tri explicite pour éviter toute incohérence chronologique.

#### Opérations réalisées :
1. **Tri séquentiel prioritaire (`sort_values`) :**
   - Réorganisation du DataFrame par `Delivery_person_ID` et `ID` de commande. Ce tri garantit que les enregistrements contigus appartiennent à la même session de livraison.
2. **Propagation contextuelle (`ffill` / `bfill`) :**
   - Application d'un remplissage avant/arrière au sein de chaque groupe de livreurs (`groupby('Delivery_person_ID')`) pour attribuer la date exacte des livraisons encadrantes.
3. **Purge de sécurité (`dropna`) :**
   - Élimination des rares lignes dont la date reste indéterminable (livreurs n'ayant aucune autre date enregistrée dans la base).

In [ ]:

if 'Order_Date' in df.columns:
    sort_cols = [c for c in ['Delivery_person_ID', 'ID'] if c in df.columns]
    if sort_cols:
        df = df.sort_values(by=sort_cols).reset_index(drop=True)
    if 'Delivery_person_ID' in df.columns:
        df['Order_Date'] = df.groupby('Delivery_person_ID')['Order_Date'].transform(
            lambda x: x.ffill().bfill()
        )
    df = df.dropna(subset=['Order_Date'])
print("Traitement de 'Order_Date' terminé. NaN restants :", df['Order_Date'].isnull().sum())

Traitement de 'Order_Date' terminé. NaN restants : 0


### 6. Imputation des variables secondaires (Numériques et Catégorielles)

**Objectif :** Finaliser le traitement des valeurs manquantes sur l'ensemble des caractéristiques secondaires de la flotte de livraison, du trafic et du contexte de commande, afin d'obtenir un jeu de données 100 % complet.

#### Opérations réalisées :
1. **Imputation des variables quantitatives (`Delivery_person_Age`, `Delivery_person_Ratings`) :**
   - Remplissage des valeurs manquantes par la **moyenne** (`.mean()`) de chaque colonne.
   - *Remarque :* Préserve la tendance centrale globale de l'âge des livreurs et de leurs évaluations.
2. **Imputation des variables catégorielles et discrètes (`Road_traffic_density`, `multiple_deliveries`, `City`, `Festival`) :**
   - Remplissage des valeurs manquantes par le **mode** (`.mode()[0]`), soit la catégorie statistique la plus fréquente dans le dataset (ex: le niveau de trafic le plus représenté ou l'absence de festival par défaut).
3. **Contrôle d'intégrité final (`df.isnull().sum()`) :**
   - Exécution d'un décompte global des `NaN` pour valider que chaque colonne affiche strictement **`0`** valeur manquante avant d'exporter le fichier vers l'Étape 2.

---

In [ ]:
other_num_to_impute = ["Delivery_person_Age", "Delivery_person_Ratings"]
for col in other_num_to_impute:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mean())  
other_cat_to_impute = ["Road_traffic_density", "multiple_deliveries", "City", "Festival"]
for col in other_cat_to_impute:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])



             City  Restaurant_latitude  Restaurant_longitude  \
5   Metropolitian            27.163303             78.057044   
14  Metropolitian            27.163303             78.057044   
42  Metropolitian            27.161661             78.011544   
43  Metropolitian            27.161661             78.011544   
44          Urban            27.161661             78.011544   

     Time_Order_picked  
5  1900-01-01 11:05:00  
14 1900-01-01 17:15:00  
42 1900-01-01 09:25:00  
43 1900-01-01 08:45:00  
44 1900-01-01 23:05:00  


In [37]:
df = df.dropna(subset=["Time_Orderd"])
print(df["Time_Orderd"].isna().sum())
df.to_csv("../data/cleaned_dataset.csv", index=False)

0
